In [1]:
import glob
import math
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Download list of Olink genes

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/preprocessed/proteomics_genes.txt

olink_genes = pl.read_csv('proteomics_genes.txt', has_header=False).rename({'column_1': 'region'})
olink_genes

[===========================================================>] Completed 46,102 of 46,102 bytes (100%) /home/dnanexus/ukbgym/utils/average_pheno_per_variant/proteomics_genes.txtt


region
str
"""ENSG00000266967"""
"""ENSG00000114779"""
"""ENSG00000097007"""
"""ENSG00000060971"""
"""ENSG00000157766"""
…
"""ENSG00000173465"""
"""ENSG00000105428"""
"""ENSG00000188372"""


In [5]:
# Download annotation file for variant-gene mapping
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym.parquet -o /home/dnanexus/data_dir/

anno = pl.read_parquet(
    '/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet', 
    columns=['id', 'region']
)

anno

Error: path "/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet"
already exists but -f/--overwrite was not set


id,region
str,str
"""chr2:26136528:A:G""","""ENSG00000084733"""
"""chr2:233771145:C:T""","""ENSG00000241119"""
"""chr12:101117325:T:G""","""ENSG00000151572"""
"""chr10:25235985:T:G""","""ENSG00000151025"""
"""chr7:80447426:C:CA""","""ENSG00000135218"""
…,…
"""chr7:90733917:A:C""","""ENSG00000058091"""
"""chr18:754318:C:G""","""ENSG00000176105"""
"""chr3:97895184:G:C""","""ENSG00000080200"""


In [6]:
anno['region'].value_counts(sort=True).join(olink_genes, on='region', how='semi')

region,count
str,u64
"""ENSG00000174469""",605106
"""ENSG00000185008""",444475
"""ENSG00000189283""",435466
"""ENSG00000021645""",385076
"""ENSG00000149972""",376497
…,…
"""ENSG00000179889""",1616
"""ENSG00000160221""",669
"""ENSG00000267368""",611


In [8]:
# Download Olink: covariates and PRS corrected
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/adjusted/cauc_cov_regression_90pcs_prs.parquet -o /home/dnanexus/data_dir/

# PROTRIDER corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/adjusted/cauc_protrider_lite_prs_rint.parquet -o /home/dnanexus/data_dir/

phenos = pl.read_parquet('/home/dnanexus/data_dir/cauc_protrider_lite_prs_rint.parquet')

olink_genes_w_data = list(set(phenos.columns).intersection(set(olink_genes['region'])).intersection(set(anno['region'])))

phenos = phenos.select(['sample'] + olink_genes_w_data)

long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=olink_genes_w_data,
        variable_name='phenotype',
        value_name='pheno_value'
    )

    .with_columns(
        phenotype = pl.col('phenotype') + '_olink'
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

[===========================================================>] Completed 1,034,847,849 of 1,034,847,849 bytes (100%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_rint.parquett
shape: (2_661, 2)
┌───────────────────────┬───────┐
│ phenotype             ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u64   │
╞═══════════════════════╪═══════╡
│ ENSG00000170323_olink ┆ 39208 │
│ ENSG00000113721_olink ┆ 39208 │
│ ENSG00000095713_olink ┆ 39208 │
│ ENSG00000172023_olink ┆ 39208 │
│ ENSG00000069702_olink ┆ 39208 │
│ …                     ┆ …     │
│ ENSG00000102837_olink ┆ 31597 │
│ ENSG00000131050_olink ┆ 31502 │
│ ENSG00000111405_olink ┆ 30953 │
│ ENSG00000163131_olink ┆ 30432 │
│ ENSG00000170373_olink ┆ 29106 │
└───────────────────────┴───────┘


sample,phenotype,pheno_value
str,str,f64
"""5645319""","""ENSG00000074706_olink""",-0.200664
"""5959139""","""ENSG00000074706_olink""",0.942694
"""5673208""","""ENSG00000074706_olink""",-0.186023
"""5732867""","""ENSG00000074706_olink""",0.444875
"""2074480""","""ENSG00000074706_olink""",-0.000882
…,…,…
"""1807196""","""ENSG00000107201_olink""",2.932203
"""4223555""","""ENSG00000107201_olink""",-0.636716
"""2992773""","""ENSG00000107201_olink""",0.588168


In [9]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(pl.col('gt')==1)
)
long_gt.head().collect()

[===========================================================>] Completed 27,974,638,284 of 27,974,638,284 bytes (100%) /home/dnanexus/data_dir/gt_long.parquett>                                                         ] Downloaded 1,308,622,848 of 27,974,638,284 bytes (4%) /home/dnanexus/data_dir/gt_long.parquet[===========>                                                ] Downloaded 5,637,144,576 of 27,974,638,284 bytes (20%) /home/dnanexus/data_dir/gt_long.parquet


id,sample,gt
str,str,i8
"""chr10:20020006:C:T""","""5317620""",1
"""chr10:20020007:TTTTCTTGC:T""","""5546988""",1
"""chr10:20020010:T:C""","""2793793""",1
"""chr10:20020014:G:A""","""1722739""",1
"""chr10:20020014:G:A""","""1139673""",1


In [10]:
output_dir = '/home/dnanexus/data_dir/appv_phenos/'
!mkdir -p {output_dir}

CHUNK_SIZE = 100
total_genes = len(olink_genes_w_data)
num_chunks = math.ceil(total_genes / CHUNK_SIZE)

print(f"Processing {total_genes} genes in {num_chunks} chunks...")

# Process in Batches
for i in tqdm(range(0, total_genes, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_genes = olink_genes_w_data[i : i + CHUNK_SIZE]
    chunk_phenos = [f"{g}_olink" for g in chunk_genes]

# for olink_gene in tqdm(olink_genes_w_data):
    print(f"Processing chunk starting at index: {i}")
    
    (
        anno.filter(pl.col('region').is_in(chunk_genes))
        .lazy()
        .join(
            long_gt,
            on='id',
            how='inner'
        )

        .join(
            long_phenos.filter(pl.col('phenotype').is_in(chunk_phenos)).lazy(),
            on='sample',
            # on=['sample', 'phenotype'],
            how='inner'
        )

        .group_by(['id', 'region', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )

        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(method="max")
                .over("region")
                .cast(pl.Float32)
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len()
            ).cast(pl.Float32)
        )

        .sink_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk{i}.parquet')
        # .collect(engine='streaming')
    )

Processing 2661 genes in 27 chunks...


  0%|          | 0/27 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  4%|▎         | 1/27 [00:23<10:17, 23.74s/it]

Processing chunk starting at index: 100


...

 93%|█████████▎| 25/27 [10:06<00:45, 22.90s/it]

Processing chunk starting at index: 2500


 96%|█████████▋| 26/27 [10:30<00:23, 23.07s/it]

Processing chunk starting at index: 2600


100%|██████████| 27/27 [10:48<00:00, 24.01s/it]


## Consolidate parquet

In [11]:
pl.read_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk0.parquet')

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:26451540:C:T""","""ENSG00000077420""","""ENSG00000105289_olink""",13,0.063828,0.856842,573801.0,0.010414
"""chr10:26451962:A:G""","""ENSG00000077420""","""ENSG00000091986_olink""",6,0.174372,1.28291,643510.0,0.011679
"""chr10:26451999:A:G""","""ENSG00000077420""","""ENSG00000108179_olink""",6,0.394891,1.00323,760548.0,0.013803
"""chr10:26451314:C:G""","""ENSG00000077420""","""ENSG00000137809_olink""",2,0.142181,0.350257,624009.0,0.011325
"""chr10:26451314:C:G""","""ENSG00000077420""","""ENSG00000090659_olink""",2,-0.770253,0.988105,166136.0,0.003015
…,…,…,…,…,…,…,…
"""chr10:71553502:T:G""","""ENSG00000107736""","""ENSG00000050327_olink""",1,1.039484,null,3.009201e6,0.054612
"""chr10:71563729:CTTTTTTTTTCTTTT…","""ENSG00000107736""","""ENSG00000115602_olink""",1,0.380208,null,2.363268e6,0.042889
"""chr10:71549352:A:G""","""ENSG00000107736""","""ENSG00000150551_olink""",1,-0.631936,null,655544.0,0.011897


In [12]:
combined_output_file = "/home/dnanexus/data_dir/olink_protrider_prs_rint_all_genes_EURunrelated_appv_percentiles.parquet"

# 1. Get list of files manually
files = glob.glob(output_dir)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(combined_output_file, engine='streaming')
print("Done.")

Found 1 files.
Streaming to disk...
Done.


In [13]:
!dx upload {combined_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 27,347,224,016 of 27,347,224,016 bytes (100%) /home/dnanexus/data_dir/olink_protrider_prs_rint_all_genes_EURunrelated_appv_percentiles.parquet===============================================>            ] Uploaded 21,843,935,232 of 27,347,224,016 bytes (80%) /home/dnanexus/data_dir/olink_protrider_prs_rint_all_genes_EURunrelated_appv_percentiles.parquet=======>                                                   ] Uploaded 3,925,868,544 of 27,347,224,016 bytes (14%) /home/dnanexus/data_dir/olink_protrider_prs_rint_all_genes_EURunrelated_appv_percentiles.parquet[==========>                                                 ] Uploaded 4,831,838,208 of 27,347,224,016 bytes (18%) /home/dnanexus/data_dir/olink_protrider_prs_rint_all_genes_EURunrelated_appv_percentiles.parquet[=================>                                          ] Uploaded 8,153,726,976 of 27,347,224,016 bytes (30%) /home/dnanexus/data_dir/olink_protri

In [14]:
a = pl.scan_parquet(combined_output_file)
a.head().collect()

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:26451540:C:T""","""ENSG00000077420""","""ENSG00000105289_olink""",13,0.063828,0.856842,573801.0,0.010414
"""chr10:26451962:A:G""","""ENSG00000077420""","""ENSG00000091986_olink""",6,0.174372,1.28291,643510.0,0.011679
"""chr10:26451999:A:G""","""ENSG00000077420""","""ENSG00000108179_olink""",6,0.394891,1.00323,760548.0,0.013803
"""chr10:26451314:C:G""","""ENSG00000077420""","""ENSG00000137809_olink""",2,0.142181,0.350257,624009.0,0.011325
"""chr10:26451314:C:G""","""ENSG00000077420""","""ENSG00000090659_olink""",2,-0.770253,0.988105,166136.0,0.003015


In [15]:
a.select(['region']).collect()['region'].unique()

region
str
"""ENSG00000118046"""
"""ENSG00000116473"""
"""ENSG00000115839"""
"""ENSG00000065675"""
"""ENSG00000134460"""
…
"""ENSG00000151929"""
"""ENSG00000131943"""
"""ENSG00000110492"""
